# 🌳 Ternary-Bonsai-2-27B (Q2_0) on Colab — llama.cpp + secured public API

Run cells top to bottom. At the end you get a **public HTTPS URL + API key + ready-to-paste OpenCode config**.

- Model: `prism-ml/Ternary-Bonsai-2-27B-gguf` **Q2_0** (~7.6 GB) — 100% in T4 VRAM
- Runner: PrismML-fork **prebuilt** `llama-server` (stock llama.cpp can't run these ternary tensors on CUDA)
- Served model id: `bonsai-2-27b`

### About the 40–50 tok/s target
Decode speed on a GPU is limited by memory bandwidth: **T4 = 320 GB/s ÷ 7.6 GB weights ≈ 42 tok/s theoretical ceiling**, so a plain run lands around 25–32 tok/s.
To push past that, cell 6 **auto-tests speculative decoding** (n-gram drafting, great for code) and keeps whichever config is fastest. Results depend on how repetitive the output is (editing existing code gains most).
An L4 is similar to a T4; an A100 would easily exceed 50 tok/s.

> ⚠️ Reasoning model: give requests generous `max_tokens`, or set `THINKING = False` in cell 1 for faster coding replies.
> ⚠️ Keep the tab open (free Colab disconnects on idle). Tunnel URL changes each run; the API key protects it.

In [ ]:
# 1) Check GPU + settings (edit these)
import subprocess, shutil, secrets, json

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
assert gpu, 'No GPU found: Runtime > Change runtime type > T4 GPU'
print('GPU:', gpu)
g = shutil.disk_usage('/')
print(f'disk: {g.free/1e9:.0f} GB free (need ~15 GB)')

CFG = {
    'CTX': 65536,       # 64K context (native 262k). T4 VRAM math: ~7.6 GB weights + ~64 KiB/token
                     # KV (fp16) -> 128K needs an 8 GB KV cache and OOMs. With the q8_0 KV
                     # flags in cell 4 (~32 KiB/token) 64K = ~2 GB KV -> fits easily; 131072
                     # would be ~4 GB KV -> tight but possible. Measure with cell 5's VRAM print.
    'THINKING': True,    # False = skip reasoning, faster/shorter answers for coding
    'API_KEY': secrets.token_urlsafe(16),   # protects the public tunnel
    'PORT': 11434,
    'EXTRA': [],         # extra server flags (cell 6 may fill this with the fastest config)
}
json.dump(CFG, open('/content/config.json', 'w'))
print('config saved')

GPU: Tesla T4, 15360 MiB, 580.82.07
disk: 121 GB free (need ~15 GB)
config saved


In [ ]:
# 2) Download prebuilt llama-server (PrismML fork) + CUDA runtime libs. No compile.
import subprocess, re, glob, os, site, json, urllib.request

out = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True).stdout
m = re.search(r'Driver Version:\s*(\d+)\.', out)
major = int(m.group(1)) if m else 0

TAG = 'prism-b12128-d7b2896'
FORK = f'https://github.com/PrismML-Eng/llama.cpp/releases/download/{TAG}/'
STOCK = 'https://github.com/ggml-org/llama.cpp/releases/download/b12359/'

# Forced to the most compatible CUDA build
asset = 'llama-prism-b12128-d7b2896-bin-linux-cuda-12.4-x64.tar.gz'
print(f'NVIDIA driver {major}.x -> {asset}')

subprocess.run(f'curl -fsSL -o /content/{asset} {FORK}{asset}', shell=True, check=True)
subprocess.run(f'rm -rf /content/pkg && mkdir -p /content/pkg && tar -xzf /content/{asset} -C /content/pkg', shell=True, check=True)
found = glob.glob('/content/pkg/**/llama-server', recursive=True)
assert found, 'llama-server not found in archive'
found = found[0]
os.chmod(found, 0o755)

# CUDA runtime libs (Colab often lacks cudart/cuBLAS; without them the server silently runs on CPU)
r = subprocess.run(f'curl -fsSL -o /content/cudart.tar.gz {STOCK}cudart-llama-b12359-bin-ubuntu-cuda-12.8-x64.tar.gz', shell=True)
subprocess.run('rm -rf /content/stock && mkdir -p /content/stock', shell=True)
if r.returncode == 0:
    subprocess.run('tar -xzf /content/cudart.tar.gz -C /content/stock', shell=True, check=True)
else:
    print('cudart bundle download failed, falling back to pip CUDA 12 libs')

# Always install the pip CUDA 12 libs too, as a second source of the libraries
subprocess.run('pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12', shell=True, check=True)

ld = {os.path.dirname(x) for x in glob.glob('/content/pkg/**/lib*.so*', recursive=True)}
ld |= {os.path.dirname(x) for x in glob.glob('/content/stock/**/lib*.so*', recursive=True)}
for sd in site.getsitepackages():
    ld |= {os.path.dirname(x) for x in glob.glob(os.path.join(sd, 'nvidia', '*', 'lib', 'lib*.so*'))}

envline = f'LD_LIBRARY_PATH={":".join(sorted(ld))}' if ld else ''
open('/content/llama_bin.txt', 'w').write(found)
open('/content/llama_env.txt', 'w').write(envline)
print('llama-server:', found)

# Check that the CUDA backend can actually find its libraries
d = os.path.dirname(found)
for so in glob.glob(d + '/libggml-cuda*'):
    missing = subprocess.run(f'{envline} ldd {so} | grep "not found"', shell=True, capture_output=True, text=True).stdout
    print(os.path.basename(so), '->', missing.strip() or 'all libraries found')
print(subprocess.run(f'{envline} "{found}" --list-devices 2>&1 | head -10', shell=True, capture_output=True, text=True).stdout)

NVIDIA driver 580.x -> llama-prism-b12128-d7b2896-bin-linux-cuda-12.4-x64.tar.gz
llama-server: /content/pkg/llama-prism-b12128-d7b2896/llama-server
libggml-cuda.so -> libcuda.so.1 => not found
Available devices:
  (none)



In [ ]:
import subprocess, glob, os

# 1) Locate the real driver library (libcuda.so.1)
cands = glob.glob('/usr/lib64-nvidia/libcuda.so*') + glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*') \
      + glob.glob('/usr/local/nvidia/lib64/libcuda.so*')
print('libcuda candidates:', cands)
assert cands, 'libcuda.so.1 not found: check the runtime has a GPU (Runtime > Change runtime type > T4)'
drv_dirs = sorted({os.path.dirname(c) for c in cands})

# 2) Rebuild LD_LIBRARY_PATH: CUDA libs + driver dir + Colab's existing value
old = open('/content/llama_env.txt').read().strip().split('=', 1)[-1]
parts = [p for p in old.split(':') if p] + drv_dirs + [p for p in os.environ.get('LD_LIBRARY_PATH', '').split(':') if p]
seen, paths = set(), []
for p in parts:
    if p not in seen:
        seen.add(p); paths.append(p)
envline = 'LD_LIBRARY_PATH=' + ':'.join(paths)
open('/content/llama_env.txt', 'w').write(envline)

# 3) Verify
LLAMA = open('/content/llama_bin.txt').read().strip()
d = os.path.dirname(LLAMA)
for so in glob.glob(d + '/libggml-cuda*'):
    miss = subprocess.run(f'{envline} ldd {so} | grep "not found"', shell=True, capture_output=True, text=True).stdout
    print(os.path.basename(so), '->', miss.strip() or 'all libraries found')
print(subprocess.run(f'{envline} "{LLAMA}" --list-devices 2>&1 | head -10', shell=True, capture_output=True, text=True).stdout)

libcuda candidates: ['/usr/lib64-nvidia/libcuda.so.1', '/usr/lib64-nvidia/libcuda.so.580.82.07', '/usr/lib64-nvidia/libcuda.so']
libggml-cuda.so -> all libraries found
Available devices:
  CUDA0: Tesla T4 (14912 MiB, 14807 MiB free)



In [ ]:
# 3) Download the model (7.63 GB). Re-run to resume if interrupted.
import subprocess, os
URL = 'https://huggingface.co/prism-ml/Ternary-Bonsai-2-27B-gguf/resolve/main/Ternary-Bonsai-2-27B-Q2_0.gguf'
subprocess.run(f'wget -c -q --show-progress -O /content/Ternary-Bonsai-2-27B-Q2_0.gguf {URL}', shell=True, check=True)
size = os.path.getsize('/content/Ternary-Bonsai-2-27B-Q2_0.gguf') / 1e9
print(f'model size: {size:.2f} GB')
assert size > 7.0, 'Download incomplete, re-run this cell to resume'

model size: 7.63 GB


In [ ]:
# 4) Server helpers + start llama-server (everything on GPU)
import subprocess, time, json, urllib.request, urllib.error, os

CFG = json.load(open('/content/config.json'))
LLAMA = open('/content/llama_bin.txt').read().strip()
ENV = open('/content/llama_env.txt').read().strip()
BASE = f"http://127.0.0.1:{CFG['PORT']}"
os.makedirs('/content/logs', exist_ok=True)

def log_tail(n=2500):
    return open('/content/logs/llama-server.log', errors='ignore').read()[-n:]

def start_server(extra=(), wait=420):
    subprocess.run('pkill -f llama-server; sleep 3; true', shell=True)
    args = [
        '-m /content/Ternary-Bonsai-2-27B-Q2_0.gguf', '--alias bonsai-2-27b',
        '-ngl 99', '-fa on',
        '-ctk q8_0', '-ctv q8_0',   # half-size KV cache: ~32 KiB/token instead of 64
        f"-c {CFG['CTX']}",
        '-np 1',                    # single slot: whole context for one session (fastest, no splitting)
        '-b 2048', '-ub 1024',      # faster prompt processing for big agent prompts
        '--cache-reuse 256',        # reuse KV cache between agent turns
        '--jinja',                  # chat template + tool calling (needed by OpenCode)
        '--temp 1.0 --top-p 0.95 --top-k 20 --min-p 0.05',
        f"--api-key {CFG['API_KEY']}",
        f"--host 127.0.0.1 --port {CFG['PORT']}",
    ]
    if not CFG['THINKING']:
        args.append('--reasoning-budget 0')
    args += list(extra)
    cmd = f'{ENV} nohup "{LLAMA}" ' + ' '.join(args) + ' > /content/logs/llama-server.log 2>&1 &'
    subprocess.Popen(['bash', '-c', cmd])
    t0 = time.time()
    while time.time() - t0 < wait:
        time.sleep(3)
        if subprocess.run('pgrep -f llama-server', shell=True, capture_output=True).returncode != 0:
            print(log_tail()); raise RuntimeError('llama-server exited, see log above')
        try:
            if urllib.request.urlopen(BASE + '/health', timeout=5).status == 200:
                print(f'server ready in {time.time()-t0:.0f}s  (flags: {" ".join(extra) or "baseline"})')
                return
        except Exception:
            pass
    print(log_tail()); raise RuntimeError('llama-server did not become healthy in time')

def api(path, body, timeout=900):
    req = urllib.request.Request(BASE + path, data=json.dumps(body).encode(),
        headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ' + CFG['API_KEY']})
    return json.loads(urllib.request.urlopen(req, timeout=timeout).read())

start_server(CFG['EXTRA'])

server ready in 166s  (flags: baseline)


In [ ]:
# 5) Smoke test + GPU check + speed benchmark
import subprocess

def gpu_used():
    o = subprocess.run('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits',
                       shell=True, capture_output=True, text=True).stdout.strip()
    return int(o.splitlines()[0])

used = gpu_used()
print(f'VRAM used: {used} MiB')
assert used > 5000, 'Model is NOT on the GPU (silent CPU fallback). Check log:\n' + log_tail(1500)

out = api('/v1/chat/completions', {'model': 'bonsai-2-27b', 'max_tokens': 400,
          'messages': [{'role': 'user', 'content': 'Reply with exactly: MODEL LOADED'}]})
print(out['choices'][0]['message']['content'][:200])

PROMPT = ('# Python 3\n# LRU cache with get/put and O(1) operations, plus unit tests\n'
          'from collections import OrderedDict\n\nclass LRUCache:\n')

def bench(n=256, runs=3):
    api('/completion', {'prompt': PROMPT, 'n_predict': 32, 'cache_prompt': False})  # warm-up
    speeds = []
    for _ in range(runs):
        r = api('/completion', {'prompt': PROMPT, 'n_predict': n, 'temperature': 0.2,
                                'cache_prompt': False, 'ignore_eos': True})
        speeds.append(r['timings']['predicted_per_second'])
    avg = sum(speeds) / len(speeds)
    print(f'decode: {avg:.1f} tok/s  (runs: {", ".join(f"{s:.1f}" for s in speeds)})')
    return avg

BASELINE = bench()

In [ ]:
# 6) AUTO-TUNE: try speculative decoding (n-gram drafting, no draft model needed).
#    Keeps the fastest config. If a flag isn't supported by this build, it is skipped automatically.
CANDIDATES = [
    ['--spec-type', 'ngram-mod', '--spec-ngram-size-n', '24', '--draft-min', '16', '--draft-max', '48'],
    ['--spec-type', 'ngram-simple', '--draft-min', '8', '--draft-max', '32'],
]
best_speed, best_extra = BASELINE, []
for extra in CANDIDATES:
    try:
        start_server(extra)
        s = bench()
        # also verify output is still sane
        txt = api('/completion', {'prompt': PROMPT, 'n_predict': 40, 'temperature': 0, 'cache_prompt': False})['content']
        if len(txt.strip()) > 10 and s > best_speed * 1.05:
            best_speed, best_extra = s, extra
    except Exception as e:
        print('candidate skipped:', str(e)[:200])

CFG['EXTRA'] = best_extra
json.dump(CFG, open('/content/config.json', 'w'))
print(f'\nBEST: {best_speed:.1f} tok/s with flags: {" ".join(best_extra) or "baseline"}')
start_server(best_extra)     # restart on the winning config
if best_speed < 40:
    print('Below 40 tok/s: this is the T4 memory-bandwidth limit (~42 tok/s ceiling). '
          'Spec decoding helps most on repetitive/edit-heavy code. Try THINKING=False or an L4/A100 runtime.')

In [ ]:
# 7) Public tunnel (Cloudflare quick tunnel, no account) + ready-to-paste OpenCode config
import subprocess, time, re, json

CFG = json.load(open('/content/config.json'))
subprocess.run('pkill -f cloudflared; true', shell=True)
subprocess.run('curl -fsSL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared', shell=True, check=True)
subprocess.Popen(['bash', '-c',
    f"/content/cloudflared tunnel --url http://127.0.0.1:{CFG['PORT']} --no-autoupdate > /content/logs/tunnel.log 2>&1"])
url = None
for _ in range(40):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/logs/tunnel.log', errors='ignore').read())
    if m:
        url = m.group(0); break
if not url:
    print(open('/content/logs/tunnel.log', errors='ignore').read()[-800:])
    raise RuntimeError('tunnel did not come up')
time.sleep(5)  # let the edge register the tunnel
CFG['URL'] = url
json.dump(CFG, open('/content/config.json', 'w'))

print('PUBLIC URL :', url)
print('OpenAI base:', url + '/v1')
print('API key    :', CFG['API_KEY'])
print('Model id   : bonsai-2-27b\n')
opencode = {
    '$schema': 'https://opencode.ai/config.json',
    'provider': {'colab': {
        'npm': '@ai-sdk/openai-compatible', 'name': 'Bonsai 2 27B (Colab)',
        'options': {'baseURL': url + '/v1', 'apiKey': CFG['API_KEY']},
        'models': {'bonsai-2-27b': {'name': 'Bonsai 2 27B',
                                    'limit': {'context': CFG['CTX'], 'output': 8192}}}}}}
print('--- paste into opencode.json (~/.config/opencode/opencode.json) ---')
print(json.dumps(opencode, indent=2))

In [ ]:
# 8) Verify the public URL end-to-end
import json, urllib.request
CFG = json.load(open('/content/config.json'))
req = urllib.request.Request(CFG['URL'] + '/v1/chat/completions',
    data=json.dumps({'model': 'bonsai-2-27b', 'max_tokens': 400,
                     'messages': [{'role': 'user', 'content': 'Reply with exactly: API IS LIVE'}]}).encode(),
    headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ' + CFG['API_KEY'],
             'User-Agent': 'curl/8.0'})   # Cloudflare blocks the default Python user-agent
out = json.loads(urllib.request.urlopen(req, timeout=600).read())
print(out['choices'][0]['message']['content'][:300])
print('\nTimings:', {k: round(v, 1) for k, v in out.get('timings', {}).items() if k.endswith('per_second')})

In [ ]:
# 9) (Optional) Keep-alive monitor: keeps the session busy and restarts server/tunnel if they die.
#    Leave this cell running; stop it with the square button.
import time, subprocess, urllib.request, json
CFG = json.load(open('/content/config.json'))
while True:
    time.sleep(60)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{CFG['PORT']}/health", timeout=10)
    except Exception:
        print('server down, restarting...'); start_server(CFG['EXTRA'])
    if subprocess.run('pgrep -f cloudflared', shell=True, capture_output=True).returncode != 0:
        print('tunnel down: re-run cell 7 (URL will change)'); break

## 🔌 Connect OpenCode (or any OpenAI client)
Cell 7 prints a complete `opencode.json` block with your URL and API key — merge it into your OpenCode config and pick **Bonsai 2 27B (Colab)**.
Other clients: `base_url = <URL>/v1`, `api_key = <printed key>`, model `bonsai-2-27b`.

### Notes
- Session restart = new VM: re-run cells 1→8 (binary ~2 min, model ~3 min). New tunnel URL and new API key each time.
- If OpenCode says **Invalid API Key** — or you need the URL/key again — run the last cell (**FIX THE API**):
  it restarts the server with the key from `config.json`, keeps the running tunnel URL, and prints both.
- `CTX` defaults to 64k (native 262k). KV cache costs ~32 KiB/token with the q8_0 flags baked into
  the server; 128k (~4 GB KV) is possible but tight on a T4, and huge prompts process slowly anyway.
- Recommended sampling for thinking mode is already baked into the server flags.
- If tok/s is low, check `nvidia-smi` shows ~8+ GB used and that the runtime is a T4/L4/A100, not CPU.

In [ ]:
# 10) FIX THE API — run this whenever OpenCode says “Invalid API Key”
#     (or whenever you need the URL + key printed again).
#     The server only accepts the exact key it was started with. This cell:
#       - shows the CURRENT server args (so you can see a key mismatch)
#       - restarts llama-server with the key from /content/config.json
#         and 128K context (same tuned flags as cell 4)
#       - keeps the running tunnel alive  ->  your URL does NOT change
#       - verifies the call end-to-end through the public URL, then
#         prints everything to paste into the setup .bat / OpenCode
import subprocess, time, urllib.request, json, re, os
CFG = json.load(open('/content/config.json'))
API_KEY = CFG['API_KEY']
# keep the CTX from config.json — do NOT raise it here: 128K fp16 KV OOMs a T4
BASE = f"http://127.0.0.1:{CFG['PORT']}"
LLAMA = open('/content/llama_bin.txt').read().strip()
ENV = open('/content/llama_env.txt').read().strip()

# 1) EVIDENCE: what args is the current server running with?
pid = subprocess.run('pgrep -f llama-server | head -1', shell=True,
                     capture_output=True, text=True).stdout.strip()
if pid:
    print('CURRENT ARGS:', open(f'/proc/{pid}/cmdline', 'rb')
          .read().decode().replace('\x00', ' ')[:400])

# 2) restart WITH the known key + 128K context
parts = [
    f'"{LLAMA}"', '-m /content/Ternary-Bonsai-2-27B-Q2_0.gguf', '--alias bonsai-2-27b',
    '-ngl 99', '-fa on', '-ctk q8_0', '-ctv q8_0', f"-c {CFG['CTX']}",
    '-np 1', '-b 2048', '-ub 1024', '--cache-reuse 256', '--jinja',
    '--temp 1.0 --top-p 0.95 --top-k 20 --min-p 0.05',
    f'--api-key {API_KEY}', f"--host 127.0.0.1 --port {CFG['PORT']}",
]
if not CFG.get('THINKING', True):
    parts.append('--reasoning-budget 0')
cmd = f'{ENV} nohup ' + ' '.join(parts) + ' > /content/logs/llama-server.log 2>&1 &'
subprocess.run('pkill -f llama-server; sleep 3; true', shell=True)
subprocess.Popen(['bash', '-c', cmd])
t0 = time.time()
while time.time() - t0 < 420:
    time.sleep(3)
    try:
        if urllib.request.urlopen(BASE + '/health', timeout=5).status == 200:
            print(f"server ready in {time.time()-t0:.0f}s (CTX {CFG['CTX']}, q8_0 KV, secured with your key)")
            break
    except Exception:
        pass
else:
    print(open('/content/logs/llama-server.log', errors='ignore').read()[-2500:])
    raise RuntimeError('server did not restart')
json.dump(CFG, open('/content/config.json', 'w'))

# 3) tunnel: reuse the running one (URL unchanged) or start a fresh one
url = None
tlog = open('/content/logs/tunnel.log', errors='ignore').read() if os.path.exists('/content/logs/tunnel.log') else ''
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tlog)
alive = subprocess.run('pgrep -f "cloudflared tunnel"', shell=True).returncode == 0
if m and alive:
    url = m.group(0)
    print('reusing the running tunnel — URL unchanged')
else:
    subprocess.run('pkill -f cloudflared; true', shell=True)
    if not os.path.exists('/content/cloudflared'):
        subprocess.run('curl -fsSL -o /content/cloudflared '
                       'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
                       '&& chmod +x /content/cloudflared', shell=True, check=True)
    subprocess.Popen(['bash', '-c',
        f"/content/cloudflared tunnel --url http://127.0.0.1:{CFG['PORT']} --no-autoupdate "
        '> /content/logs/tunnel.log 2>&1'])
    for _ in range(40):
        time.sleep(2)
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',
                      open('/content/logs/tunnel.log', errors='ignore').read())
        if m:
            url = m.group(0); break
assert url, 'tunnel did not come up — check /content/logs/tunnel.log'
CFG['URL'] = url
json.dump(CFG, open('/content/config.json', 'w'))

# 4) prove it works through the public URL with the key
req = urllib.request.Request(
    url + '/v1/chat/completions',
    data=json.dumps({'model': 'bonsai-2-27b', 'max_tokens': 8,
                     'messages': [{'role': 'user', 'content': 'hi'}]}).encode(),
    headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ' + API_KEY,
             'User-Agent': 'curl/8.0'})   # Cloudflare blocks the default Python user-agent
out = json.loads(urllib.request.urlopen(req, timeout=900).read())
print('end-to-end OK through the public URL')

print()
print('PUBLIC URL :', url)
print('OpenAI base:', url + '/v1')
print('API key    :', API_KEY)
print('Model id   : bonsai-2-27b')
print('-> run the setup .bat: paste the URL, choose 2, paste this key')
